In [1]:

import importlib
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
import functions_mlp
importlib.reload(functions_mlp)
from notebooks.OOF.functions_mlp import train_profile_mlp, predict_profile_mlp


import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from pathlib import Path


In [2]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1363
Liczba etykiet: 1362


In [3]:
data_processed_volume = np.load(
    project_dir / "data" / "processed_volumes_with_rotation_by_similarity.npz",
    allow_pickle=False,
)

X_volumes = data_processed_volume["X"].astype(np.float32)
y_volumes = data_processed_volume["y"].astype(np.int64)
uids_processed_volumes = data_processed_volume["uid"]

data_profiles = np.load(
    project_dir / "data" / "processed_profiles.npz",
    allow_pickle=False,
)

print(data_profiles.files)

uids_profiles = data_profiles["uid"]
y_profiles = data_profiles["is_pathologic"].astype(np.int64)

x_profile = data_profiles["x_profile"]
y_profile = data_profiles["y_profile"]
z_profile = data_profiles["z_profile"]
rotation_x_profile = data_profiles["rotation_x_profile"]
tilt_x_profiles = data_profiles["tilt_x_profiles"]
pitch_y_profiles = data_profiles["pitch_y_profiles"]

print("Volumes:", X_volumes.shape, y_volumes.shape, uids_processed_volumes.shape)
print("Profiles:", y_profiles.shape, uids_profiles.shape)

print("UID order equal:", np.array_equal(uids_processed_volumes, uids_profiles))
print("Labels equal:", np.array_equal(y_volumes, y_profiles))

assert np.array_equal(uids_processed_volumes, uids_profiles)
assert np.array_equal(y_volumes, y_profiles)

uid_to_idx = {
    uid: idx
    for idx, uid in enumerate(uids_processed_volumes)
}

['uid', 'is_pathologic', 'x_profile', 'y_profile', 'z_profile', 'rotation_x_profile', 'tilt_x_profiles', 'pitch_y_profiles']
Volumes: (1362, 1, 50, 60, 30) (1362,) (1362,)
Profiles: (1362,) (1362,)
UID order equal: True
Labels equal: True


In [4]:
split_df = pd.DataFrame({
    "uid": uids_processed_volumes,
    "is_pathologic": y_volumes,
})

train_main_df, temp_df = train_test_split(
    split_df,
    test_size=0.3,
    stratify=split_df["is_pathologic"],
    random_state=42,
)

valid_meta_df, test_final_df = train_test_split(
    temp_df,
    test_size=1/3,
    stratify=temp_df["is_pathologic"],
    random_state=42,
)

train_main_df = train_main_df.reset_index(drop=True)
valid_meta_df = valid_meta_df.reset_index(drop=True)
test_final_df = test_final_df.reset_index(drop=True)

train_main_df["split"] = "train_main"
valid_meta_df["split"] = "valid_meta"
test_final_df["split"] = "test_final"

train_main_df["fold"] = -1
valid_meta_df["fold"] = -1
test_final_df["fold"] = -1

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (_, val_idx) in enumerate(
    skf.split(
        train_main_df["uid"],
        train_main_df["is_pathologic"],
    )
):
    train_main_df.loc[val_idx, "fold"] = fold

oof_split_df = pd.concat(
    [
        train_main_df,
        valid_meta_df,
        test_final_df,
    ],
    ignore_index=True,
)

oof_split_df.to_csv(
    "oof_split.csv",
    index=False,
)

print(oof_split_df["split"].value_counts())
print(pd.crosstab(oof_split_df["split"], oof_split_df["is_pathologic"]))
print(pd.crosstab(train_main_df["fold"], train_main_df["is_pathologic"]))

assert oof_split_df["uid"].is_unique
assert set(train_main_df["fold"]) == set(range(5))

split
train_main    953
valid_meta    272
test_final    137
Name: count, dtype: int64
is_pathologic    0    1
split                  
test_final      62   75
train_main     430  523
valid_meta     123  149
is_pathologic   0    1
fold                  
0              86  105
1              86  105
2              86  105
3              86  104
4              86  104


In [5]:
print(oof_split_df.groupby("split")["is_pathologic"].mean())
print(train_main_df.groupby("fold")["is_pathologic"].mean())

split
test_final    0.547445
train_main    0.548793
valid_meta    0.547794
Name: is_pathologic, dtype: float64
fold
0    0.549738
1    0.549738
2    0.549738
3    0.547368
4    0.547368
Name: is_pathologic, dtype: float64


In [6]:
X_profiles = np.concatenate(
    [
        x_profile,
        y_profile,
        z_profile,
        rotation_x_profile,
    ],
    axis=1,
).astype(np.float32)
def add_result(results, model_name, fold, y_true, y_proba):
    results.append({
        "model": model_name,
        "fold": fold,
        "log_loss": log_loss(y_true, y_proba),
        "auroc": roc_auc_score(y_true, y_proba),
    })



In [7]:

results = []
for fold in range(5):
    print(f"\nFold {fold}")

    train_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] != fold),
        "uid"
    ].to_numpy()

    valid_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] == fold),
        "uid"
    ].to_numpy()

    train_idx = np.array([uid_to_idx[uid] for uid in train_uids])
    valid_idx = np.array([uid_to_idx[uid] for uid in valid_uids])

    y_train = y_volumes[train_idx]
    y_valid = y_volumes[valid_idx]

    print("train:", len(train_idx), "valid:", len(valid_idx))

    profile_logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=10000,
        random_state=42,
        solver="saga",
    )
    )

    param_grid = {
    "logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
    "logisticregression__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
}

    profile_logreg_grid = GridSearchCV(
        estimator=profile_logreg_pipeline,
        param_grid=param_grid,
        scoring="neg_log_loss",
        cv=3,
        n_jobs=-1,
    )

    profile_logreg_grid.fit(
        X_profiles[train_idx],
        y_train,
    )

    best_profile_logreg = profile_logreg_grid.best_estimator_

    joblib.dump(
        best_profile_logreg,
        f"models/profile_logreg_fold_{fold}.joblib",
    )

    p_profile_logreg = best_profile_logreg.predict_proba(
        X_profiles[valid_idx]
    )[:, 1]

    add_result(
        results=results,
        model_name="profile_logreg",
        fold=fold,
        y_true=y_valid,
        y_proba=p_profile_logreg,
    )

results



Fold 0
train: 762 valid: 191

Fold 1
train: 762 valid: 191

Fold 2
train: 762 valid: 191

Fold 3
train: 763 valid: 190

Fold 4
train: 763 valid: 190


[{'model': 'profile_logreg',
  'fold': 0,
  'log_loss': 0.41439083218574524,
  'auroc': 0.8942414174972314},
 {'model': 'profile_logreg',
  'fold': 1,
  'log_loss': 0.41586676239967346,
  'auroc': 0.8880398671096346},
 {'model': 'profile_logreg',
  'fold': 2,
  'log_loss': 0.4103754162788391,
  'auroc': 0.8955703211517165},
 {'model': 'profile_logreg',
  'fold': 3,
  'log_loss': 0.5313894152641296,
  'auroc': 0.8339669051878354},
 {'model': 'profile_logreg',
  'fold': 4,
  'log_loss': 0.4981141686439514,
  'auroc': 0.8353085867620752}]

In [8]:
results_df = pd.DataFrame(results)

summary_df = results_df.groupby("model").agg(
    log_loss_mean=("log_loss", "mean"),
    log_loss_std=("log_loss", "std"),
    auroc_mean=("auroc", "mean"),
    auroc_std=("auroc", "std"),
)

display(results_df)
display(summary_df)

results_df.to_csv(
    "results/profile_logreg_oof_results.csv",
    index=False,
)

,model,fold,log_loss,auroc
0,profile_logreg,0,0.414391,0.894241
1,profile_logreg,1,0.415867,0.888040
2,profile_logreg,2,0.410375,0.895570
3,profile_logreg,3,0.531389,0.833967
4,profile_logreg,4,0.498114,0.835309


,log_loss_mean,log_loss_std,auroc_mean,auroc_std
model,,,,
profile_logreg,0.454027,0.056704,0.869425,0.031887


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [11]:
mlp_results = []
def add_result_mlp(
    results,
    model_name,
    fold,
    y_true,
    y_proba,
    params=None,
    model_path=None,
):
    row = {
        "model": model_name,
        "fold": fold,
        "log_loss": log_loss(y_true, y_proba),
        "auroc": roc_auc_score(y_true, y_proba),
    }

    if params is not None:
        row["params"] = str(params)

    if model_path is not None:
        row["model_path"] = str(model_path)

    results.append(row)

model_profile_mlp, best_valid_loss = train_profile_mlp(
        X_train=X_profiles[train_idx],
        y_train=y_train,
        X_valid=X_profiles[valid_idx],
        y_valid=y_valid,
        input_size=X_profiles.shape[1],
        epochs=300,
        batch_size=64,
        learning_rate=1e-4,
        weight_decay=1e-4,
        patience=30,
    )

model_path = f"models/profile_mlp_fold_{fold}.pth"

torch.save(
        model_profile_mlp.state_dict(),
        model_path,
)

p_profile_mlp = predict_profile_mlp(
        model_profile_mlp,
        X_profiles[valid_idx],
)

add_result_mlp(
        results=mlp_results,
        model_name="profile_mlp",
        fold=fold,
        y_true=y_valid,
        y_proba=p_profile_mlp,
        params={
            "best_valid_loss": best_valid_loss,
            "model_path": str(model_path),
        },
    )

print(mlp_results[-1])



{'model': 'profile_mlp', 'fold': 4, 'log_loss': 0.4825846552848816, 'auroc': 0.8472719141323792, 'params': "{'best_valid_loss': 0.482584685087204, 'model_path': 'models/profile_mlp_fold_4.pth'}"}


In [ ]:
functions_mlp.__file__
